# 03. Adam Optimizer Demystified: From SGD to Adaptive Moments

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/The-Pocket/PocketFlow-Tutorial-Video-Generator/blob/main/docs/llm/03_Adam_Optimizer_Demystified.ipynb)

> **Tutorial Overview**: Understand why standard SGD struggles in ravines and saddle points, how Momentum solves oscillations, how RMSprop scales learning rates, and how Adam combines both with Bias Correction.
> **Original Source**: `adam.md`

---


In [ ]:
# Setup & Imports
!pip install -q torch matplotlib numpy

import torch
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)


# **Title: Give me 30 min, I will make the Adam Optimizer Click. Forever.**

## Intro

You know Gradient Descent. You know the formula: `new_weight = old_weight - learning_rate * gradient`. It's simple. It works.

But it's not what powers modern AI.

In every state-of-the-art model, in every high-performance training script, you see the same name: **Adam**. You're told to just use it. It's the default. It's "better."

But why?

You look up the algorithm and are hit with a wall of math. A black box of Greek letters and strange terms that feel impossibly complex.

*   Exponentially Weighted Moving Average (EWMA)
*   First and Second Moments
*   Bias Correction

It seems like a magic spell you're supposed to cast without understanding.

Here's the secret: **Adam isn't one complex idea. It's three simple ideas, bolted together to solve three specific problems.**

Give me 30 minutes. We will tear Adam down to its fundamental parts and rebuild it from the ground up. No skipped steps. No magic. By the end, you will have a deep, intuitive, and permanent understanding of every single component.

This is the algorithm you are about to master:

1.  **First Moment (Momentum):** $m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$
2.  **Second Moment (Adaptive LR):** $v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$
3.  **Bias Correction:** $\hat{m}_t = \frac{m_t}{1 - \beta_1^t}$ and $\hat{v}_t = \frac{v_t}{1 - \beta_2^t}$
4.  **The Update:** $\theta_t = \theta_{t-1} - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$

That wall of math will look simple. You will see *why* each piece exists and the exact problem it solves.

Ready to make it click? Let's begin.

## **Part 1: The First Problem - Inefficient Progress**

**The Core Problem:** Gradient Descent is memoryless. As it gets closer to the minimum and the gradient shrinks, its steps become smaller and smaller, causing it to slow down dramatically.

Let's use the simple function `f(p) = p**2`. The minimum is at `p=0`, and the gradient is `f'(p) = 2p`. We start at `p=10` and use a learning rate of `η = 0.1`.

#### **Algorithm 1: Standard Gradient Descent**

The update rule is simple and direct.


```python
FOR each iteration:
  gradient = 2 * params
  params = params - 0.1 * gradient
```


This is our baseline—the slow, steady crawl.

| Iteration | Current `p` | Gradient `g = 2p` | Update `0.1 * g` | New `p` |
| :-------- | :---------- | :---------------- | :--------------- | :------ |
| 0         | 10.000      | 20.000            | 2.000            | 8.000   |
| 1         | 8.000       | 16.000            | 1.600            | 6.400   |
| 2         | 6.400       | 12.800            | 1.280            | 5.120   |
| 3         | 5.120       | 10.240            | 1.024            | 4.096   |
| 4         | 4.096       | 8.192             | 0.819            | 3.277   |

**Analysis of the Slowness:** The "Update" size is constantly shrinking: `2.0` → `1.6` → `1.28`... This is **deceleration**. The algorithm becomes less effective with every step.

#### **Algorithm 2: Gradient Descent with Momentum**

Now, let's add a `velocity` term with a more moderate `beta` of `0.5`. This will allow inertia to build up without running out of control.
